# 09 残差连接与 LayerNorm

进入 Transformer Encoder 之前，还需要补两个非常重要的结构：

```text
残差连接 Residual Connection
LayerNorm 层归一化
```

在 Transformer Encoder 里，一个子层通常不是简单地写成：

```text
x -> Sublayer(x)
```

而是会配合残差连接和 LayerNorm。

先给一句核心直觉：

```text
残差连接负责保留原信息，让网络更容易训练。
LayerNorm 负责稳定每个 token 的特征分布。
```

这一节先讲清楚概念和形状，不写代码。

## 1. 为什么进入 Encoder 前要补这两个东西

我们已经学了：

```text
Multi-Head Attention
Feed Forward Network
位置编码
```

但标准 Transformer Encoder Layer 里还会出现：

```text
Add
Norm
```

Add 通常指残差相加。

Norm 通常指 LayerNorm。

如果不先补这两个概念，进入 Encoder 时会看到：

```text
Multi-Head Attention -> Add & Norm -> FFN -> Add & Norm
```

但不知道 Add 和 Norm 为什么在这里。

所以这一节就是给完整 Encoder 铺最后一块地板。

## 2. 先说残差连接是什么

普通子层可以写成：

$$
\mathbf{y}=F(\mathbf{x})
$$

这里 $F$ 可以是 Multi-Head Attention，也可以是 FFN。

残差连接会把原输入 $\mathbf{x}$ 加回去：

$$
\mathbf{y}=\mathbf{x}+F(\mathbf{x})
$$

这就是残差连接最核心的形式。

可以用一句话理解：

```text
子层不必从零开始生成全部输出，只需要学习在原输入基础上应该改多少。
```

## 3. 残差连接的直觉：保留原始信息

假设某个 token 当前表示是 $\mathbf{x}$。

经过 Attention 或 FFN 后得到：

$$
F(\mathbf{x})
$$

如果直接用 $F(\mathbf{x})$ 替换 $\mathbf{x}$，原来的信息可能会被改得太狠。

残差连接让输出变成：

$$
\mathbf{x}+F(\mathbf{x})
$$

也就是说：

```text
原来的信息保留一份。
子层新学到的变化再加上去。
```

这很适合 Transformer。

因为 token 表示在很多层里会不断被改写，残差连接可以帮助信息不要在层层变换中轻易丢失。

## 4. 残差连接的另一个直觉：学习增量

残差连接也可以理解成学习增量。

普通网络像是在学：

```text
最终输出应该是什么。
```

残差结构更像是在学：

```text
在原输入基础上，我应该补充或修改什么。
```

如果某一层暂时没学到有用变化，它可以让 $F(\mathbf{x})$ 接近 0。

这样：

$$
\mathbf{x}+F(\mathbf{x})\approx\mathbf{x}
$$

也就是这层至少可以近似保持输入不变。

这让深层网络更容易训练。

因为模型不必担心每一层都必须做复杂改变。

## 5. 残差连接为什么要求形状一致

残差连接要做加法：

$$
\mathbf{x}+F(\mathbf{x})
$$

能相加的前提是形状一致。

比如输入是：

```text
B x N x D
```

那么子层输出也应该是：

```text
B x N x D
```

这样才能逐元素相加：

```text
B x N x D + B x N x D -> B x N x D
```

这也解释了为什么前面学 FFN 时，最后要从 $d_{ff}$ 压回 $D$。

如果不压回 $D$，残差连接就加不上了。

## 6. 在 Attention 子层里怎么加残差

假设输入是：

```text
X：B x N x D
```

经过 Multi-Head Attention 后得到：

```text
MHA(X)：B x N x D
```

残差连接就是：

$$
X+\operatorname{MHA}(X)
$$

形状是：

```text
B x N x D + B x N x D -> B x N x D
```

含义是：

```text
保留原来的 token 表示。
再加上 Attention 从上下文里汇总得到的新信息。
```

## 7. 在 FFN 子层里怎么加残差

FFN 子层也是同样的逻辑。

假设 Attention 子层之后的输入记作 $H$：

```text
H：B x N x D
```

FFN 输出也是：

```text
FFN(H)：B x N x D
```

残差连接：

$$
H+\operatorname{FFN}(H)
$$

形状仍然是：

```text
B x N x D
```

含义是：

```text
保留 FFN 前的表示。
再加上 FFN 加工出来的变化量。
```

## 8. 为什么深层网络喜欢残差连接

Transformer 通常会堆很多层。

如果每一层都完全重写输入表示，训练会很困难。

残差连接提供了一条比较直接的信息通路。

信息和梯度都更容易穿过很多层。

可以先这样理解：

```text
没有残差：每一层都必须完全接住上一层的输出再重新变换。
有残差：原信息可以沿着加法通路继续往后传。
```

所以残差连接是深层网络能稳定训练的重要原因之一。

## 9. 接下来讲 LayerNorm 是什么

残差连接解决了“保留原信息、帮助深层训练”的问题。

但在深层网络里，还有另一个问题：

```text
每一层输出的数值分布可能不断变化。
```

如果数值太大、太小、分布不稳定，训练就会变得困难。

归一化层的作用就是让数值分布更稳定。

Transformer 里最常用的是 LayerNorm。

它的核心思想是：

```text
对每个样本、每个 token 的特征维度做归一化。
```

## 10. LayerNorm 对谁做归一化

假设输入形状是：

```text
B x N x D
```

其中：

- $B$ 表示 batch size。
- $N$ 表示 token 数量。
- $D$ 表示每个 token 的特征维度。

LayerNorm 通常对最后一维 $D$ 做归一化。

也就是说，对每个 token 的 D 个特征单独计算均值和方差。

可以想成：

```text
第 1 个样本的第 1 个 token：对它自己的 D 个特征归一化。
第 1 个样本的第 2 个 token：对它自己的 D 个特征归一化。
第 2 个样本的第 1 个 token：对它自己的 D 个特征归一化。
...
```

LayerNorm 不需要拿整个 batch 的统计量来归一化。

## 11. 用一个 token 看 LayerNorm

假设某个 token 的向量是 4 维：

```text
x = [2, 4, 6, 8]
```

先算这个 token 自己 4 个特征的均值：

$$
\mu=\frac{2+4+6+8}{4}=5
$$

再看每个值和均值的差：

```text
[-3, -1, 1, 3]
```

再用标准差缩放。

简化理解就是让这个 token 的特征分布变得更稳定：

```text
均值接近 0
方差接近 1
```

真实 LayerNorm 还会有可学习的缩放和平移参数，让模型可以调整归一化后的分布。

## 12. LayerNorm 的公式

对一个 token 向量 $\mathbf{x}$，LayerNorm 可以写成：

$$
\operatorname{LayerNorm}(\mathbf{x})=\gamma\frac{\mathbf{x}-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta
$$

这里：

- $\mu$ 是这个 token 各个特征的均值。
- $\sigma^2$ 是这个 token 各个特征的方差。
- $\epsilon$ 是一个很小的数，防止除以 0。
- $\gamma$ 是可学习缩放参数。
- $\beta$ 是可学习平移参数。

先不要被公式吓住。

它的主线就是：

```text
先标准化，再允许模型学习缩放和平移。
```

## 13. 为什么 LayerNorm 后形状不变

LayerNorm 改变的是数值分布，不改变形状。

输入是：

```text
B x N x D
```

输出仍然是：

```text
B x N x D
```

因为它只是对每个 token 的 D 个特征做标准化和缩放平移。

不会改变 token 数量。

也不会改变每个 token 的维度。

所以：

```text
LayerNorm 稳定数值，不改变结构形状。
```

## 14. LayerNorm 和 BatchNorm 有什么区别

你前面学过 BatchNorm。

BatchNorm 通常依赖 batch 维度上的统计量。

它会看一批样本在某些通道或特征上的均值和方差。

LayerNorm 不一样。

LayerNorm 对每个样本、每个 token 自己的特征维度做归一化。

简单对比：

| 方法 | 主要沿什么方向统计 | 对 batch size 是否敏感 | Transformer 中常见程度 |
|---|---|---|---|
| BatchNorm | batch 维或通道统计 | 比较敏感 | 较少用于标准 Transformer |
| LayerNorm | 单个样本的特征维 | 不依赖 batch 统计 | 非常常见 |

Transformer 处理序列时，batch 大小、序列长度、mask 等情况比较复杂。

LayerNorm 不依赖 batch 统计，因此更适合这类结构。

## 15. Add & Norm 是什么意思

在 Transformer 图里，经常看到：

```text
Add & Norm
```

Add 指残差相加。

Norm 指 LayerNorm。

一种经典写法是：

$$
\operatorname{LayerNorm}(\mathbf{x}+\operatorname{Sublayer}(\mathbf{x}))
$$

读成中文就是：

```text
先让子层处理 x。
再把原来的 x 加回去。
最后做 LayerNorm。
```

这里的 Sublayer 可以是：

```text
Multi-Head Attention
FFN
```

## 16. 在 Encoder 里有两次 Add & Norm

一个 Encoder Layer 里通常有两个子层。

第一个子层是 Multi-Head Self-Attention。

第二个子层是 FFN。

所以会有两次 Add & Norm。

可以先写成：

```text
第一段：
X -> Multi-Head Attention -> Add & Norm

第二段：
上一步输出 -> FFN -> Add & Norm
```

更具体一点：

$$
H=\operatorname{LayerNorm}(X+\operatorname{MHA}(X))
$$

$$
O=\operatorname{LayerNorm}(H+\operatorname{FFN}(H))
$$

这就是经典 Post-Norm 形式的直观写法。

## 17. Pre-Norm 和 Post-Norm 先只做概念了解

你以后可能会看到两种写法。

Post-Norm：

$$
\operatorname{LayerNorm}(x+\operatorname{Sublayer}(x))
$$

Pre-Norm：

$$
x+\operatorname{Sublayer}(\operatorname{LayerNorm}(x))
$$

区别是 LayerNorm 放在子层前还是子层后。

入门阶段不要在这里钻太深。

先知道：

```text
Post-Norm：先子层和残差，再 Norm。
Pre-Norm：先 Norm，再子层和残差。
```

现代实现里经常会使用 Pre-Norm 变体，因为训练深层模型时更稳定。

但理解 Encoder 结构时，先掌握 Add、Norm 各自作用即可。

## 18. 残差连接和 LayerNorm 的分工

现在把两者放在一起看。

残差连接解决的是：

```text
原始信息怎么保留下来。
深层网络怎么更容易传递信息和梯度。
```

LayerNorm 解决的是：

```text
每层输出的数值分布怎么更稳定。
每个 token 的特征尺度怎么更容易控制。
```

所以 Add & Norm 可以理解成：

```text
先把原信息和子层新信息合并。
再把合并后的表示整理到更稳定的数值状态。
```

## 19. 常见误解 1：残差连接只是简单多加一次输入

从公式看，残差连接确实是加法。

但它不只是为了多加一点数值。

它改变了网络学习的方式。

普通子层学习完整输出：

```text
我要把 x 变成 y。
```

残差结构更像学习变化量：

```text
我在 x 的基础上补充什么。
```

这就是它对深层网络训练很有帮助的原因。

## 20. 常见误解 2：LayerNorm 会改变 token 数量

不会。

LayerNorm 不会改变 token 数量，也不会改变向量维度。

输入：

```text
B x N x D
```

输出还是：

```text
B x N x D
```

它只是对每个 token 的特征数值做标准化和可学习缩放平移。

所以 LayerNorm 改变的是数值分布，不是结构形状。

## 21. 常见误解 3：LayerNorm 和 Softmax 类似

LayerNorm 和 Softmax 都会处理一组数字。

但它们完全不是一回事。

Softmax 的作用是：

```text
把一组分数变成非负、总和为 1 的权重分布。
```

LayerNorm 的作用是：

```text
把一个 token 的特征做标准化，让均值和方差更稳定。
```

Softmax 常用于注意力权重。

LayerNorm 常用于稳定隐藏表示。

不要把它们混成同一种归一化。

## 22. 本节小结

这一节先记住：

1. 残差连接的核心形式是 $x+F(x)$。
2. 残差连接可以保留原始信息，让子层学习增量。
3. 残差连接要求输入和子层输出形状一致。
4. Transformer 子层通常保持 `B x N x D`，这让残差相加变得方便。
5. LayerNorm 对每个样本、每个 token 的特征维度做归一化。
6. LayerNorm 不依赖 batch 统计，适合 Transformer。
7. LayerNorm 输入输出形状不变，仍然是 `B x N x D`。
8. Add & Norm 通常表示残差相加加 LayerNorm。
9. Encoder Layer 里通常有两次 Add & Norm。
10. Pre-Norm 和 Post-Norm 是 LayerNorm 放置位置不同，先知道概念即可。

## 23. 自测问题

1. 为什么进入 Transformer Encoder 前要先学习残差连接和 LayerNorm？
2. 残差连接的核心公式是什么？
3. 为什么可以把残差连接理解成学习增量？
4. 为什么残差连接要求输入和子层输出形状一致？
5. 在 Attention 子层中，`X + MHA(X)` 分别表示什么？
6. LayerNorm 主要解决什么问题？
7. 对 `B x N x D` 的输入，LayerNorm 通常沿哪一维做归一化？
8. LayerNorm 和 BatchNorm 的关键区别是什么？
9. Add & Norm 中的 Add 和 Norm 分别指什么？
10. Encoder Layer 里为什么通常有两次 Add & Norm？
11. Pre-Norm 和 Post-Norm 的区别是什么？
12. LayerNorm 和 Softmax 有什么区别？